In [1]:
"""
Generate a dictionary of all KEGG human (hsa) pathways with descriptive keys
and store it as a .txt file.

Output format matches:
{
    "MAPK_signaling": "hsa04010",
    ...
}
"""

import requests
import re

KEGG_URL = "https://rest.kegg.jp/list/pathway/hsa"
OUTPUT_FILE = "hsa_pathways.txt"


def normalize_key(name: str) -> str:
    """
    Convert KEGG pathway name to a descriptive, Python-style key
    similar to the example provided by the user.
    """
    name = name.replace(" - Homo sapiens (human)", "")
    name = name.replace(" signaling pathway", " signaling")
    name = name.replace(" pathway", "")
    name = name.replace(" disease", "")
    name = name.replace(" beta ", " beta ")

    # replace special chars
    name = name.replace("β", "beta")
    name = name.replace("κ", "k")
    name = name.replace("/", " ")
    name = name.replace("-", " ")

    # collapse whitespace and underscores
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", "_", name.strip())

    return name


def fetch_hsa_pathways() -> dict:
    response = requests.get(KEGG_URL)
    response.raise_for_status()

    pathway_dict = {}

    for line in response.text.strip().split("\n"):
        pathway_id, pathway_name = line.split("\t")
        key = normalize_key(pathway_name)
        pathway_dict[key] = pathway_id

    return pathway_dict


pathway_id_dict = fetch_hsa_pathways()
import json

with open("hsa_pathways.json", "w") as f:
    json.dump(pathway_id_dict, f, indent=2, sort_keys=True)

